In [ ]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

import time
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0)
chip.adc.set_gap(adc_cs_gap=90,adc_first_gap=10,adc_last_gap=10)
chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
chip.clk_manager.set_cyc(10, 50)
chip.add_compiler("../compiler/code/")

In [ ]:
set_good_device_file_name = "../chip_data/chip8_/cond_500.npy"
reset_good_device_file_name = "../chip_data/chip8_/cond_250.npy"

# 1.set操作

In [ ]:
pos = (0,256,0,256)
chip.write_point3(*pos,write_voltage=3,tg=1.6,pulse_width=1e-6,set_device=True)

In [ ]:
pos = (0,256,0,256)
voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
plot_cond(cond,title="cond",vmin=0,vmax=1500)

# 2.reset操作

In [ ]:
# # 对某一块区域内的点进行reset
# pos = (0,256,0,256)
# chip.write_point3(*pos,write_voltage=3,tg=1.6,pulse_width=1e-6,set_device=False)

In [ ]:
def Reset(write_times,start_v,delta_v,tg,threshold,reset_pulse_width,read_type=2,sub_base=False,vmax=1000):
    need_read = np.ones((256,256),dtype=bool)
    voltage_base = np.zeros((256,256))
    voltage = np.zeros((256,256))
    cond_sub_base = None
    for i in range(write_times):
        print(f"write_time = {i}")
        v = start_v+i*delta_v
        if read_type == 2:
            if sub_base:
                voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
            voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        elif read_type == 3:
            pos = (0,256,0,256)
            if sub_base:
                voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
            voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
        if sub_base:
            cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        else:
            cond_sub_base = chip.voltage_to_cond(voltage)

        condition_reset = cond_sub_base>threshold
        need_read = condition_reset

        plot_cond(cond_sub_base,title=f"v={v:.2f}-needReset={np.sum(condition_reset)}",vmax=vmax)

        chip.write_point2(crossbar=condition_reset,write_voltage=v,tg=tg,pulse_width=reset_pulse_width,set_device=False)

In [ ]:
Reset(21,start_v=1,delta_v=0.05,tg=5,threshold=200,reset_pulse_width=100e-6,read_type=2,sub_base=True)

In [ ]:
pos = (0,256,0,256)
voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
print(np.sum(cond<250))
np.save(reset_good_device_file_name,cond<250)

In [ ]:
cond = chip.voltage_to_cond(voltage=voltage-voltage_base)-700
plot_cond(cond,vmin=-500,vmax=500)

interval = 5
bin_edges = np.linspace(-1000, 1000, int(2000/interval)+1)  
data = cond.flatten()
counts, bin_edges, _ = plt.hist(data, bins=bin_edges, color='blue', alpha=0.7, edgecolor='None')

# 3.Forming器件

In [ ]:
# # 对某一块区域内的点进行set
# pos = (0,256,0,256)
# chip.write_point3(*pos,write_voltage=3,tg=1.6,pulse_width=1e-6,set_device=True)

In [ ]:
def Forming(write_times,write_voltage,start_tg,delta_tg,threshold,set_pulse_width,read_type=2,sub_base=False):
    need_read = np.ones((256,256),dtype=bool)
    voltage_base = np.zeros((256,256))
    voltage = np.zeros((256,256))
    cond_sub_base = None
    for i in range(write_times):
        print(f"write_time = {i}")
        tg = start_tg+i*delta_tg
        if read_type == 2:
            if sub_base:
                voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
            voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        elif read_type == 3:
            pos = (0,256,0,256)
            if sub_base:
                voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
            voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
        if sub_base:
            cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        else:
            cond_sub_base = chip.voltage_to_cond(voltage)



        condition_set = cond_sub_base<threshold
        need_read = condition_set

        plot_cond(cond_sub_base,title=f"tg={tg:.2f}-needForming={np.sum(condition_set)}",vmax=1200)
        # if i==0:
        #     chip.write_point2(crossbar=condition_set,write_voltage=5,tg=2.5,pulse_width=1e-6,set_device=False)
        # set的点
        chip.write_point2(crossbar=condition_set,write_voltage=write_voltage,tg=tg,pulse_width=set_pulse_width,set_device=True)

In [ ]:
chip.ps.set_time_out(10)
for i in range(1):
    Forming(write_times=1,write_voltage=2+i*0.5,start_tg=1,delta_tg=0.05,threshold=550,set_pulse_width=100e-6,read_type=2,sub_base=True)

In [ ]:
pos = (0,256,0,256)
voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
print(np.sum(cond>500))
np.save(set_good_device_file_name,cond>500)

# 3.寻找好的区域

In [ ]:
import random
def find_good_device(row,col,rownum,colnum,good_cond,cnt=10):
    random.shuffle(row)
    random.shuffle(col)

    select_row,select_col=row[:rownum],col[:colnum]
    rowtmp,coltmp=row[rownum:],col[colnum:]
    
    ansrow=[]
    anscol=[]

    for n in range(cnt):
        flag = False

        
        # 遍历所有选择的行，替换
        for j in select_row:
            k=j
            for i in rowtmp:
                if np.sum(good_cond[i,select_col])>np.sum(good_cond[k,select_col]):
                    k=i
            if k!=j:
                rowtmp.remove(k)
                rowtmp.append(j)
                flag=True
            ansrow.append(k)
        
        select_row=ansrow
        ansrow=[]

        # 遍历所有选择的列，替换
        for j in select_col:
            k=j
            for i in coltmp:
                if np.sum(good_cond[select_row,i])>np.sum(good_cond[select_row,k]):
                    k=i
            if k!=j:
                coltmp.remove(k)
                coltmp.append(j)
                flag=True
            anscol.append(k)
        
        select_col=anscol
        anscol=[]

        if not flag:
            break
        # print(n)
    return select_row,select_col

In [ ]:
cond_200 = np.load(reset_good_device_file_name)
cond_500 = np.load(set_good_device_file_name)
good_cond = (cond_200&cond_500).astype(int) + (~cond_500).astype(int)*0.2 + (~cond_200).astype(int)*-0.2
plot_cond(good_cond,vmax=1)
print(np.sum(good_cond))

In [ ]:
row=[i for i in range(5,150)]
col=[i for i in range(256)]

ansrow,anscol=find_good_device(row,col,100,100,good_cond=good_cond)

sub_matrix = good_cond[np.ix_(ansrow, anscol)]
print(np.sort(ansrow))
print(np.sort(anscol))
print(np.sum(sub_matrix))
print(np.sum(sub_matrix>0.5))
plot_cond(sub_matrix,vmax=1)

# good_cond[np.ix_(ansrow, anscol)]=0

In [ ]:
np.savez("../data/hnn/weight_pos_100_100_chip_8_2222",row=np.sort(ansrow),col=np.sort(anscol))

In [ ]:
ans = []

In [ ]:
k=0
for i in range(20):
    row=[i for i in range(0,256)]
    col=[i for i in range(256)]
    select_row=[]
    ansrow,anscol=find_good_device(row,col,49,7,good_cond=good_cond)

    print(np.sort(ansrow))
    print(np.sort(anscol))
    print(np.sum(sub_matrix))
    plot_cond(sub_matrix,vmax=1)
    if np.sum(sub_matrix)==7*49:
        ans.append((ansrow,anscol))
        np.savez(f"../data/hnn/hnn_weight_49_7_{k}",row=np.sort(ansrow),col=np.sort(anscol))
        k=k+1
    good_cond[np.ix_(ansrow, anscol)]=False

In [ ]:
print(len(ans))
np.savez("../data/hnn/weight_pos_100_100_chip_8_",np.array(ans))

In [ ]:

row=[i for i in range(0,256)]
col=[i for i in range(256)]
select_row=[]
ansrow,anscol=find_good_device(row,col,49,5,good_cond=good_cond)

sub_matrix = good_cond[np.ix_(ansrow, anscol)]
print(np.sort(ansrow))
print(np.sort(anscol))
print(np.sum(sub_matrix))
plot_cond(sub_matrix,vmax=1)

In [ ]:
np.savez("../data/hnn/hnn_weight_pos_r_1_5_7_7",row=np.sort(ansrow),col=np.sort(anscol))
# a=np.load("../data/svm/svm_weight_pos.npz")
# print(a["row"])

In [ ]:
print(np.ix_(row,col)[1])

In [ ]:
pos0 = np.zeros((256,256))
pos0[np.ix_(row,col)]=1
plot_cond(pos0,vmax=1)

good_device = np.load("../data/good_point_100_100_1_4.npy")
print(good_device[1])
pos1 = np.zeros((256,256))
pos1[np.ix_(good_device[0],good_device[1])]=1
plot_cond(pos1,vmax=1)

print(np.sum((pos0>0)&(pos1>0)))
plot_cond((pos0>0.5)&(pos1>0.5),vmax=1)